In [25]:
import torch as t
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
from typing import Literal
from enum import Enum
import yaml
import pandas as pd
import sys
import os
import json
from openai import OpenAI
sys.path.append("../")
from src.utils import inverse_likert, list_to_str
device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [37]:
class OpenaiResponse(BaseModel):
    response: str

In [93]:
with open('../configs/generation_config.yaml', 'r') as file:
    generation_config = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_questions.yaml', 'r') as file:
    question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/paraphrased_hexaco_100_questions.yaml', 'r') as file:
    paraphrased_question_list = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_eval.yaml', 'r') as file:
    hexaco_eval = yaml.safe_load(file)

In [5]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device),
    AutoTokenizer.from_pretrained(MODEL_NAME)
)

In [6]:
openai_model_name = "gpt-4.1-mini"
openai_model = outlines.from_openai(OpenAI(), openai_model_name)

In [7]:
NO_ANSWER = "Do not wish to answer"
likert_scale = generation_config['likert_scale'].copy()
likert_scale.append(NO_ANSWER)

In [8]:
inverted_likert = inverse_likert(generation_config['likert_scale'].copy())
inverted_likert.append(NO_ANSWER)

In [9]:
hexaco_template = outlines.Template.from_string("""
<|im_start>user
Task: Answer the below questions:

{{ text }}

Answer the question as either {{ likert_scale }}.
<|im_end>
<|im_start>assistant
""")


In [28]:
print(prompt)

<|im_start>user
Task: Answer the below questions:

I would be quite bored by a visit to an art gallery.

Answer the question as either strongly disagree, disagree, neutral, agree, strongly agree.
<|im_end>
<|im_start>assistant


In [10]:
likert_scale

['Strongly Disagree',
 'Disagree',
 'Neutral',
 'Agree',
 'Strongly Agree',
 'Do not wish to answer']

In [45]:
def ollama_generation(model, question, likert_scale):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale))
    answer = model(
                    prompt,
                    Literal[*likert_scale]
            )
    return answer

def openai_generation(model, question, likert_scale):
    prompt = hexaco_template(text=question, likert_scale = ", ".join(likert_scale))
    prompt = f"{prompt}, use the json format."
    
    answer = openai_model(prompt, OpenaiResponse)
    return json.loads(answer)['response']
    

def generate_answers(generation_function, model, question_list, likert_scale):
    
    answers = []
    for question in question_list:
        answer = generation_function(model, question, likert_scale)
        answers.append(answer)
    
    return answers

In [83]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [46]:
print(prompt)

<|im_start>user
Task: Answer the below questions:

I would be quite bored by a visit to an art gallery.

Answer the question as either Strongly Disagree, Disagree, Neutral, Agree, Strongly Agree, Do not wish to answer.
<|im_end>
<|im_start>assistant


### Normal Questions, Normal Likert

In [47]:
normal_hexaco_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, likert_scale)

In [61]:
write_to_json(normal_hexaco_answers_gpt_41_mini, "normal_hexaco_answers_gpt_41_mini.json")

In [48]:
normal_hexaco_answers_llama_3_2b_it = generate_answers(ollama_generation, model, question_list, likert_scale)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [62]:
write_to_json(normal_hexaco_answers_llama_3_2b_it, "normal_hexaco_answers_llama_3_2b_it.json")

### Normal Questions, Inverse_Likert

In [54]:
normal_hexaco_inverted_likert_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, question_list, inverted_likert)

In [63]:
write_to_json(normal_hexaco_inverted_likert_answers_gpt_41_mini, "normal_hexaco_inverted_likert_answers_gpt_41_mini.json")

In [55]:
normal_hexaco_inverted_likert_answers_llama_3_2b_it = generate_answers(ollama_generation, model, question_list, inverted_likert)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [64]:
write_to_json(normal_hexaco_inverted_likert_answers_llama_3_2b_it, "normal_hexaco_inverted_likert_answers_llama_3_2b_it.json")

### Paraphrase Questions, Normal Likert

In [56]:
paraphrase_hexaco_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, likert_scale)

In [65]:
write_to_json(paraphrase_hexaco_answers_gpt_41_mini, "paraphrase_hexaco_answers_gpt_41_mini.json")

In [57]:
paraphrase_hexaco_answers_llama_3_2b_it = generate_answers(ollama_generation, model, paraphrased_question_list, likert_scale)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [66]:
write_to_json(paraphrase_hexaco_answers_llama_3_2b_it, "paraphrase_hexaco_answers_llama_3_2b_it.json")

### Paraphrase Questions, Inverted Likert

In [58]:
paraphrase_hexaco_inverted_likert_answers_gpt_41_mini = generate_answers(openai_generation, openai_model, paraphrased_question_list, inverted_likert)

In [67]:
write_to_json(paraphrase_hexaco_inverted_likert_answers_gpt_41_mini, "paraphrase_hexaco_inverted_likert_answers_gpt_41_mini.json")

In [59]:
paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it = generate_answers(ollama_generation, model, paraphrased_question_list, inverted_likert)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [68]:
write_to_json(paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it, "paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it.json")

### Evaluation

In [86]:
def get_refusal_rate(answers):  
    answers = pd.Series(answers)
    refusal_answers = answers[answers == "Do not wish to answer"]
    return len(refusal_answers)/len(answers)

In [100]:
refusal_rate_dict = {}
for filename in os.listdir("base_experiment_results"):
    if "hexaco" in filename:
        answers = read_json(os.path.join("base_experiment_results",filename))
        refusal_rate = get_refusal_rate(answers)
        refusal_rate_dict[filename.split(".")[0]] = refusal_rate

In [101]:
dict(sorted(refusal_rate_dict.items()))

{'normal_hexaco_answers_gpt_41_mini': 0.01,
 'normal_hexaco_answers_llama_3_2b_it': 0.09,
 'normal_hexaco_inverted_likert_answers_gpt_41_mini': 0.02,
 'normal_hexaco_inverted_likert_answers_llama_3_2b_it': 0.11,
 'paraphrase_hexaco_answers_gpt_41_mini': 0.02,
 'paraphrase_hexaco_answers_llama_3_2b_it': 0.14,
 'paraphrase_hexaco_inverted_likert_answers_gpt_41_mini': 0.0,
 'paraphrase_hexaco_inverted_likert_answers_llama_3_2b_it': 0.16}

In [98]:
hexaco_eval['emotionality']['anxiety']['indices']

[11, 35, 59, 83]